# Module 4 • Distributional Semantics and Word Embeddings

# Lesson 24 • Pretrained Embeddings and Embedding-Based Text Classification

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 120–150 minutes

---

## Scope

This lesson explains how pretrained word embeddings are loaded, inspected,
adapted, pooled into document vectors, and used in downstream text
classification.

The notebook is self-contained. It creates a small pedagogical embedding file
locally so that every cell runs without external downloads.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain what pretrained embeddings are;
- distinguish static pretrained vectors from contextual embeddings;
- load vectors from a text file;
- validate embedding dimensions and vocabulary entries;
- measure token and type coverage;
- identify out-of-vocabulary words;
- construct mean-pooled document embeddings;
- construct TF-IDF-weighted document embeddings;
- compare sparse TF-IDF and dense embedding classifiers;
- explain frozen and trainable embedding strategies;
- initialize a classifier from pretrained vectors;
- fine-tune embeddings with gradient descent;
- evaluate class performance and classification errors;
- analyze domain mismatch and semantic coverage;
- discuss Arabic and multilingual embedding use.

## Table of Contents

1. What Are Pretrained Embeddings?
2. Static Versus Contextual Representations
3. Common Embedding File Format
4. Creating a Toy Pretrained Space
5. Loading Embeddings
6. Validating the Embedding Table
7. Similarity and Nearest Neighbors
8. Classification Dataset
9. Train-Test Split
10. Vocabulary Coverage
11. Out-of-Vocabulary Handling
12. Mean-Pooled Document Embeddings
13. TF-IDF-Weighted Document Embeddings
14. Sparse TF-IDF Baseline
15. Frozen Embedding Classifier
16. Comparing Representations
17. Frozen Versus Trainable Embeddings
18. Trainable Embedding Classifier
19. Fine-Tuning Results
20. Error Analysis
21. Domain Mismatch
22. Embedding Dimension and Memory
23. Bias and Responsible Use
24. Arabic and Multilingual Considerations
25. Reproducibility and Reporting
26. Knowledge Check
27. Exercises
28. Summary and Next Lesson

# 1. What Are Pretrained Embeddings?

**Pretrained embeddings** are vector representations learned from a corpus
before the downstream task is trained.

They can transfer distributional information to tasks with limited labeled
data.

In [ ]:
import hashlib
import math
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

pretrained_overview = pd.DataFrame(
    [
        ("Training stage", "learned before the target classifier"),
        ("Main benefit", "transfer distributional information"),
        ("Typical use", "initialize or freeze embedding layers"),
        ("Main limitation", "coverage and domain mismatch"),
    ],
    columns=["Property", "Description"],
)

pretrained_overview

Pretrained vectors may come from large general corpora, domain-specific text,
multilingual corpora, or task-related collections.

# 2. Static Versus Contextual Representations

Static embeddings assign one vector to each word type.

```text
bank → one vector
```

Contextual models generate different vectors for different token occurrences.

```text
river bank
financial bank
```

In [ ]:
representation_types = pd.DataFrame(
    [
        (
            "Static embedding",
            "one vector per word type",
            "Word2Vec, GloVe, FastText",
        ),
        (
            "Contextual embedding",
            "one vector per token occurrence",
            "Transformer encoders",
        ),
    ],
    columns=["Representation", "Behavior", "Examples"],
)

representation_types

This lesson focuses on static pretrained embeddings.

# 3. Common Embedding File Format

A common text format stores one word and its vector per line:

```text
doctor 0.12 -0.08 0.41 ...
hospital 0.07 -0.02 0.38 ...
```

Some files include a header containing vocabulary size and vector dimension.

Large public embedding files may contain millions of rows and require careful
memory management.

# 4. Creating a Toy Pretrained Space

For a fully offline demonstration, we create a small semantic vector space with
four broad domains:

- health;
- finance;
- technology;
- travel.

This is a pedagogical resource, not a replacement for real pretrained vectors.

In [ ]:
EMBEDDING_DIMENSION = 16
RANDOM_SEED = 42
generator = np.random.default_rng(RANDOM_SEED)

domain_words = {
    "health": [
        "doctor", "nurse", "patient", "hospital", "clinic",
        "treatment", "medicine", "exercise", "nutrition", "health",
        "medical", "diagnosis",
    ],
    "finance": [
        "bank", "payment", "invoice", "money", "loan",
        "interest", "account", "billing", "refund", "card",
        "price", "charge",
    ],
    "technology": [
        "software", "application", "computer", "server", "network",
        "error", "update", "install", "device", "data",
        "system", "upload",
    ],
    "travel": [
        "flight", "airport", "hotel", "tourist", "travel",
        "ticket", "luggage", "beach", "museum", "city",
        "reservation", "journey",
    ],
}

shared_words = [
    "need", "problem", "help", "please", "today",
    "service", "information", "request",
]

domain_centers = {
    domain: generator.normal(
        0.0,
        1.0,
        size=EMBEDDING_DIMENSION,
    )
    for domain in domain_words
}

toy_vectors = {}

for domain, words in domain_words.items():
    center = domain_centers[domain]

    for word in words:
        toy_vectors[word] = (
            center
            + generator.normal(
                0.0,
                0.18,
                size=EMBEDDING_DIMENSION,
            )
        )

for word in shared_words:
    toy_vectors[word] = generator.normal(
        0.0,
        0.35,
        size=EMBEDDING_DIMENSION,
    )

print("Toy vocabulary size:", len(toy_vectors))

In [ ]:
embedding_file = Path("lesson_24_toy_pretrained_vectors.txt")

with embedding_file.open(
    "w",
    encoding="utf-8",
) as file:
    file.write(
        f"{len(toy_vectors)} {EMBEDDING_DIMENSION}\n"
    )

    for word in sorted(toy_vectors):
        vector_text = " ".join(
            f"{value:.8f}"
            for value in toy_vectors[word]
        )

        file.write(
            f"{word} {vector_text}\n"
        )

print("Created:", embedding_file)
print("Size:", embedding_file.stat().st_size, "bytes")

# 5. Loading Embeddings

The loader validates the header and skips malformed rows.

In [ ]:
def load_text_embeddings(
    path: str | Path,
) -> tuple[dict[str, np.ndarray], int]:
    path = Path(path)
    vectors = {}

    with path.open(
        "r",
        encoding="utf-8",
    ) as file:
        first_line = file.readline().strip().split()

        if (
            len(first_line) == 2
            and all(item.isdigit() for item in first_line)
        ):
            expected_vocabulary = int(first_line[0])
            dimension = int(first_line[1])
        else:
            raise ValueError(
                "Expected a '<vocabulary> <dimension>' header"
            )

        for line_number, line in enumerate(file, start=2):
            parts = line.strip().split()

            if len(parts) != dimension + 1:
                continue

            word = parts[0]

            try:
                vector = np.asarray(
                    parts[1:],
                    dtype=float,
                )
            except ValueError:
                continue

            vectors[word] = vector

    if len(vectors) != expected_vocabulary:
        raise ValueError(
            "Loaded vocabulary does not match the header"
        )

    return vectors, dimension


pretrained_vectors, loaded_dimension = load_text_embeddings(
    embedding_file
)

print("Loaded words:", len(pretrained_vectors))
print("Dimension:", loaded_dimension)

# 6. Validating the Embedding Table

Useful validation checks include:

- expected dimension;
- duplicate words;
- finite values;
- zero vectors;
- vocabulary size;
- normalization policy.

In [ ]:
vector_matrix = np.vstack(
    list(pretrained_vectors.values())
)

validation = pd.Series(
    {
        "vocabulary_size": len(pretrained_vectors),
        "dimension": loaded_dimension,
        "all_finite": bool(np.isfinite(vector_matrix).all()),
        "zero_vectors": int(
            np.sum(
                np.linalg.norm(
                    vector_matrix,
                    axis=1,
                ) == 0
            )
        ),
        "mean_norm": float(
            np.linalg.norm(
                vector_matrix,
                axis=1,
            ).mean()
        ),
    },
    name="Embedding validation",
)

validation

# 7. Similarity and Nearest Neighbors

In [ ]:
def cosine_similarity_vector(
    left: np.ndarray,
    right: np.ndarray,
) -> float:
    denominator = (
        np.linalg.norm(left)
        * np.linalg.norm(right)
    )

    if denominator == 0:
        return 0.0

    return float(
        np.dot(left, right)
        / denominator
    )


similarity_pairs = [
    ("doctor", "nurse"),
    ("doctor", "hospital"),
    ("doctor", "invoice"),
    ("flight", "airport"),
    ("software", "server"),
]

for left, right in similarity_pairs:
    score = cosine_similarity_vector(
        pretrained_vectors[left],
        pretrained_vectors[right],
    )

    print(
        f"{left:<10} {right:<10} {score:.3f}"
    )

In [ ]:
def nearest_pretrained_neighbors(
    target_word: str,
    top_k: int = 6,
) -> pd.DataFrame:
    target = pretrained_vectors[target_word]
    rows = []

    for word, vector in pretrained_vectors.items():
        if word == target_word:
            continue

        rows.append(
            {
                "word": word,
                "similarity": cosine_similarity_vector(
                    target,
                    vector,
                ),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["similarity", "word"],
            ascending=[False, True],
        )
        .head(top_k)
        .reset_index(drop=True)
    )


nearest_pretrained_neighbors("doctor")

Nearest neighbors reflect the pretraining corpus and objective. They are not
definitive lexical relations.

# 8. Classification Dataset

The downstream dataset contains four balanced classes.

In [ ]:
records = [
    ("The doctor reviewed the patient diagnosis", "health"),
    ("The nurse provided medical treatment", "health"),
    ("Exercise and nutrition improve health", "health"),
    ("The clinic scheduled a patient visit", "health"),
    ("The hospital needs more medicine", "health"),
    ("A doctor discussed the treatment plan", "health"),
    ("The patient requested medical information", "health"),
    ("Nutrition and exercise support recovery", "health"),
    ("The nurse works in the hospital", "health"),
    ("The diagnosis requires specialist treatment", "health"),
    ("The clinic offers health services", "health"),
    ("The patient needs medicine today", "health"),

    ("The bank rejected the loan request", "finance"),
    ("Please send the payment invoice", "finance"),
    ("The card charge was incorrect", "finance"),
    ("I need a refund for the payment", "finance"),
    ("The account has a billing problem", "finance"),
    ("Interest increased the loan price", "finance"),
    ("The bank processed the money transfer", "finance"),
    ("The invoice contains an unexpected charge", "finance"),
    ("The card payment failed today", "finance"),
    ("Please update the billing account", "finance"),
    ("The refund request needs information", "finance"),
    ("The loan interest rate changed", "finance"),

    ("The software update caused an error", "technology"),
    ("The application cannot connect to the server", "technology"),
    ("The computer needs a system update", "technology"),
    ("The network upload failed", "technology"),
    ("Please install the application", "technology"),
    ("The device displays a server error", "technology"),
    ("The software system lost data", "technology"),
    ("The computer network is unavailable", "technology"),
    ("The upload problem started today", "technology"),
    ("The application update needs help", "technology"),
    ("The server cannot process the request", "technology"),
    ("Please install the software on the device", "technology"),

    ("The flight arrived at the airport", "travel"),
    ("The tourist booked a hotel reservation", "travel"),
    ("The luggage was missing after the flight", "travel"),
    ("We need a travel ticket today", "travel"),
    ("The tourist visited a museum in the city", "travel"),
    ("The hotel is near the beach", "travel"),
    ("The airport changed the flight time", "travel"),
    ("The journey includes a city tour", "travel"),
    ("Please update the hotel reservation", "travel"),
    ("The travel service lost my luggage", "travel"),
    ("The tourist needs airport information", "travel"),
    ("The ticket request is for a beach journey", "travel"),
]

dataset = pd.DataFrame(
    records,
    columns=["text", "label"],
)

dataset["label"].value_counts()

# 9. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    dataset["text"],
    dataset["label"],
    test_size=0.25,
    random_state=42,
    stratify=dataset["label"],
)

print("Training examples:", len(X_train))
print("Test examples:", len(X_test))

The embedding vocabulary was created independently of the train-test split. In
a real experiment, the pretrained file must not be trained on the labeled test
set.

# 10. Vocabulary Coverage

Coverage can be measured at two levels:

- **type coverage:** percentage of unique dataset tokens represented;
- **token coverage:** percentage of all token occurrences represented.

In [ ]:
TOKEN_PATTERN = re.compile(
    r"\b\w+(?:[-']\w+)*\b",
    flags=re.UNICODE,
)


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(
        text.lower()
    )


def embedding_coverage(
    texts,
    vectors,
) -> pd.Series:
    tokens = [
        token
        for text in texts
        for token in tokenize(text)
    ]

    unique_tokens = set(tokens)
    covered_tokens = [
        token
        for token in tokens
        if token in vectors
    ]
    covered_types = {
        token
        for token in unique_tokens
        if token in vectors
    }

    return pd.Series(
        {
            "token_count": len(tokens),
            "type_count": len(unique_tokens),
            "token_coverage": (
                len(covered_tokens) / len(tokens)
                if tokens else 0.0
            ),
            "type_coverage": (
                len(covered_types) / len(unique_tokens)
                if unique_tokens else 0.0
            ),
        }
    )


coverage_report = embedding_coverage(
    dataset["text"],
    pretrained_vectors,
)

coverage_report

In [ ]:
all_dataset_tokens = sorted({
    token
    for text in dataset["text"]
    for token in tokenize(text)
})

oov_types = [
    token
    for token in all_dataset_tokens
    if token not in pretrained_vectors
]

print("OOV types:", oov_types)

Coverage should be reported by domain and subgroup when these distinctions are
relevant.

# 11. Out-of-Vocabulary Handling

Common OOV strategies include:

- zero vector;
- random vector;
- learned unknown vector;
- subword-derived vector;
- character model;
- vocabulary expansion.

In [ ]:
def deterministic_subword_vector(
    word: str,
    dimension: int,
    min_n: int = 3,
    max_n: int = 5,
) -> np.ndarray:
    marked = f"<{word}>"
    ngrams = []

    for n in range(min_n, max_n + 1):
        for start in range(
            len(marked) - n + 1
        ):
            ngrams.append(
                marked[start:start + n]
            )

    if not ngrams:
        return np.zeros(dimension)

    vectors = []

    for ngram in ngrams:
        digest = hashlib.sha256(
            ngram.encode("utf-8")
        ).digest()

        seed = int.from_bytes(
            digest[:8],
            byteorder="little",
            signed=False,
        )

        local_generator = np.random.default_rng(seed)

        vectors.append(
            local_generator.normal(
                0.0,
                0.08,
                size=dimension,
            )
        )

    return np.mean(vectors, axis=0)


print(
    deterministic_subword_vector(
        "traveler",
        loaded_dimension,
    )[:5]
)

The deterministic subword function is an educational fallback. It is not a
trained FastText model.

# 12. Mean-Pooled Document Embeddings

A document can be represented by averaging its word vectors.

In [ ]:
def lookup_vector(
    word: str,
    vectors: dict[str, np.ndarray],
    dimension: int,
    oov_strategy: str = "subword",
) -> np.ndarray:
    if word in vectors:
        return vectors[word]

    if oov_strategy == "zero":
        return np.zeros(dimension)

    if oov_strategy == "subword":
        return deterministic_subword_vector(
            word,
            dimension,
        )

    raise ValueError(
        "oov_strategy must be 'zero' or 'subword'"
    )


def mean_document_vector(
    text: str,
    vectors: dict[str, np.ndarray],
    dimension: int,
    oov_strategy: str = "subword",
) -> np.ndarray:
    tokens = tokenize(text)

    if not tokens:
        return np.zeros(dimension)

    matrix = np.vstack(
        [
            lookup_vector(
                token,
                vectors,
                dimension,
                oov_strategy,
            )
            for token in tokens
        ]
    )

    return matrix.mean(axis=0)


example_vector = mean_document_vector(
    "The doctor treated the patient",
    pretrained_vectors,
    loaded_dimension,
)

print(example_vector.shape)

In [ ]:
X_train_mean = np.vstack(
    [
        mean_document_vector(
            text,
            pretrained_vectors,
            loaded_dimension,
        )
        for text in X_train
    ]
)

X_test_mean = np.vstack(
    [
        mean_document_vector(
            text,
            pretrained_vectors,
            loaded_dimension,
        )
        for text in X_test
    ]
)

print("Train matrix:", X_train_mean.shape)
print("Test matrix:", X_test_mean.shape)

Mean pooling ignores word order and assigns equal weight to every token.

# 13. TF-IDF-Weighted Document Embeddings

TF-IDF weighting gives larger contributions to informative words.

In [ ]:
weighting_vectorizer = TfidfVectorizer(
    tokenizer=tokenize,
    token_pattern=None,
    lowercase=False,
)

weighting_vectorizer.fit(X_train)

weighting_vocabulary = (
    weighting_vectorizer.vocabulary_
)
weighting_idf = weighting_vectorizer.idf_


def tfidf_weighted_document_vector(
    text: str,
    vectors: dict[str, np.ndarray],
    dimension: int,
) -> np.ndarray:
    tokens = tokenize(text)
    weighted_vectors = []
    weights = []

    for token in tokens:
        feature_index = weighting_vocabulary.get(
            token
        )

        if feature_index is None:
            weight = 1.0
        else:
            weight = float(
                weighting_idf[feature_index]
            )

        vector = lookup_vector(
            token,
            vectors,
            dimension,
            oov_strategy="subword",
        )

        weighted_vectors.append(
            weight * vector
        )
        weights.append(weight)

    if not weighted_vectors:
        return np.zeros(dimension)

    return (
        np.sum(weighted_vectors, axis=0)
        / max(sum(weights), 1e-12)
    )


X_train_weighted = np.vstack(
    [
        tfidf_weighted_document_vector(
            text,
            pretrained_vectors,
            loaded_dimension,
        )
        for text in X_train
    ]
)

X_test_weighted = np.vstack(
    [
        tfidf_weighted_document_vector(
            text,
            pretrained_vectors,
            loaded_dimension,
        )
        for text in X_test
    ]
)

X_train_weighted.shape

Weighting is fitted only on the training text to avoid evaluation leakage.

# 14. Sparse TF-IDF Baseline

In [ ]:
tfidf_baseline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                tokenizer=tokenize,
                token_pattern=None,
                lowercase=False,
                ngram_range=(1, 2),
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=42,
            ),
        ),
    ]
)

tfidf_baseline.fit(
    X_train,
    y_train,
)

tfidf_predictions = tfidf_baseline.predict(
    X_test
)

tfidf_macro_f1 = f1_score(
    y_test,
    tfidf_predictions,
    average="macro",
)

print(f"TF-IDF macro F1: {tfidf_macro_f1:.3f}")

A dense embedding method should be compared against a strong sparse baseline.

# 15. Frozen Embedding Classifier

A frozen strategy keeps pretrained vectors unchanged and trains only the
downstream classifier.

In [ ]:
mean_classifier = LogisticRegression(
    max_iter=2000,
    random_state=42,
)

mean_classifier.fit(
    X_train_mean,
    y_train,
)

mean_predictions = mean_classifier.predict(
    X_test_mean
)

mean_macro_f1 = f1_score(
    y_test,
    mean_predictions,
    average="macro",
)

weighted_classifier = LogisticRegression(
    max_iter=2000,
    random_state=42,
)

weighted_classifier.fit(
    X_train_weighted,
    y_train,
)

weighted_predictions = weighted_classifier.predict(
    X_test_weighted
)

weighted_macro_f1 = f1_score(
    y_test,
    weighted_predictions,
    average="macro",
)

print(f"Mean embedding macro F1: {mean_macro_f1:.3f}")
print(f"Weighted embedding macro F1: {weighted_macro_f1:.3f}")

# 16. Comparing Representations

In [ ]:
representation_results = pd.DataFrame(
    [
        ("Sparse TF-IDF", tfidf_macro_f1),
        ("Mean pretrained embedding", mean_macro_f1),
        ("TF-IDF-weighted embedding", weighted_macro_f1),
    ],
    columns=["Representation", "Macro F1"],
).sort_values(
    "Macro F1",
    ascending=False,
)

representation_results

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(
    representation_results["Representation"],
    representation_results["Macro F1"],
)
plt.ylim(0, 1.05)
plt.ylabel("Macro F1")
plt.title("Text Representation Comparison")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

Dense embeddings may generalize through similarity, while sparse TF-IDF can
preserve highly discriminative lexical detail.

# 17. Frozen Versus Trainable Embeddings

## Frozen

- pretrained vectors remain unchanged;
- fewer trainable parameters;
- lower overfitting risk;
- limited task adaptation.

## Trainable

- vectors are updated with task gradients;
- better task adaptation is possible;
- semantic structure may be distorted;
- more labeled data may be needed.

In [ ]:
frozen_trainable = pd.DataFrame(
    [
        (
            "Frozen",
            "classifier only",
            "lower",
            "limited",
        ),
        (
            "Trainable",
            "classifier + embedding table",
            "higher",
            "greater",
        ),
    ],
    columns=[
        "Strategy",
        "Updated parameters",
        "Overfitting risk",
        "Task adaptation",
    ],
)

frozen_trainable

# 18. Trainable Embedding Classifier

The following NumPy model initializes its embedding table from the pretrained
vectors and predicts classes from mean-pooled token embeddings.

In [ ]:
label_encoder = LabelEncoder()
y_train_ids = label_encoder.fit_transform(y_train)
y_test_ids = label_encoder.transform(y_test)

training_vocabulary = sorted({
    token
    for text in X_train
    for token in tokenize(text)
})

train_word_to_index = {
    word: index
    for index, word in enumerate(training_vocabulary)
}

initial_embedding_table = np.vstack(
    [
        lookup_vector(
            word,
            pretrained_vectors,
            loaded_dimension,
            oov_strategy="subword",
        )
        for word in training_vocabulary
    ]
)


def encode_documents(texts):
    encoded = []

    for text in texts:
        ids = [
            train_word_to_index[token]
            for token in tokenize(text)
            if token in train_word_to_index
        ]

        encoded.append(ids)

    return encoded


encoded_train = encode_documents(X_train)
encoded_test = encode_documents(X_test)

print("Trainable vocabulary:", len(training_vocabulary))

In [ ]:
def softmax(values: np.ndarray) -> np.ndarray:
    shifted = values - values.max()
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum()


def train_embedding_classifier(
    encoded_documents,
    labels,
    initial_embeddings,
    class_count,
    train_embeddings=True,
    epochs=180,
    learning_rate=0.08,
    seed=42,
):
    generator = np.random.default_rng(seed)

    embeddings = initial_embeddings.copy()
    classifier_weights = generator.normal(
        0.0,
        0.1,
        size=(
            embeddings.shape[1],
            class_count,
        ),
    )
    classifier_bias = np.zeros(class_count)
    losses = []

    order = np.arange(
        len(encoded_documents)
    )

    for epoch in range(epochs):
        generator.shuffle(order)
        total_loss = 0.0

        current_rate = learning_rate * (
            1.0
            - 0.75
            * epoch
            / max(epochs - 1, 1)
        )

        for document_index in order:
            token_ids = encoded_documents[
                document_index
            ]

            if not token_ids:
                document_vector = np.zeros(
                    embeddings.shape[1]
                )
            else:
                document_vector = embeddings[
                    token_ids
                ].mean(axis=0)

            logits = (
                document_vector
                @ classifier_weights
                + classifier_bias
            )

            probabilities = softmax(logits)
            gold_class = labels[document_index]

            total_loss -= math.log(
                max(
                    probabilities[gold_class],
                    1e-12,
                )
            )

            gradient_logits = probabilities.copy()
            gradient_logits[gold_class] -= 1.0

            weight_gradient = np.outer(
                document_vector,
                gradient_logits,
            )
            bias_gradient = gradient_logits

            document_gradient = (
                classifier_weights
                @ gradient_logits
            )

            classifier_weights -= (
                current_rate
                * weight_gradient
            )
            classifier_bias -= (
                current_rate
                * bias_gradient
            )

            if train_embeddings and token_ids:
                token_gradient = (
                    document_gradient
                    / len(token_ids)
                )

                for token_id in token_ids:
                    embeddings[token_id] -= (
                        current_rate
                        * token_gradient
                    )

        losses.append(
            total_loss
            / len(encoded_documents)
        )

    return (
        embeddings,
        classifier_weights,
        classifier_bias,
        losses,
    )


def predict_embedding_classifier(
    encoded_documents,
    embeddings,
    classifier_weights,
    classifier_bias,
):
    predictions = []
    probabilities_all = []

    for token_ids in encoded_documents:
        if not token_ids:
            document_vector = np.zeros(
                embeddings.shape[1]
            )
        else:
            document_vector = embeddings[
                token_ids
            ].mean(axis=0)

        probabilities = softmax(
            document_vector
            @ classifier_weights
            + classifier_bias
        )

        predictions.append(
            int(probabilities.argmax())
        )
        probabilities_all.append(probabilities)

    return (
        np.asarray(predictions),
        np.vstack(probabilities_all),
    )

In [ ]:
(
    frozen_table,
    frozen_weights,
    frozen_bias,
    frozen_losses,
) = train_embedding_classifier(
    encoded_train,
    y_train_ids,
    initial_embedding_table,
    class_count=len(label_encoder.classes_),
    train_embeddings=False,
    seed=42,
)

(
    tuned_table,
    tuned_weights,
    tuned_bias,
    tuned_losses,
) = train_embedding_classifier(
    encoded_train,
    y_train_ids,
    initial_embedding_table,
    class_count=len(label_encoder.classes_),
    train_embeddings=True,
    seed=42,
)

frozen_ids, _ = predict_embedding_classifier(
    encoded_test,
    frozen_table,
    frozen_weights,
    frozen_bias,
)

tuned_ids, tuned_probabilities = predict_embedding_classifier(
    encoded_test,
    tuned_table,
    tuned_weights,
    tuned_bias,
)

frozen_f1 = f1_score(
    y_test_ids,
    frozen_ids,
    average="macro",
)

tuned_f1 = f1_score(
    y_test_ids,
    tuned_ids,
    average="macro",
)

print(f"Frozen neural-style classifier F1: {frozen_f1:.3f}")
print(f"Fine-tuned embedding classifier F1: {tuned_f1:.3f}")

# 19. Fine-Tuning Results

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    frozen_losses,
    label="Frozen embeddings",
)
plt.plot(
    tuned_losses,
    label="Trainable embeddings",
)
plt.xlabel("Epoch")
plt.ylabel("Average cross-entropy loss")
plt.title("Frozen and Trainable Embedding Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
embedding_shift = np.linalg.norm(
    tuned_table
    - initial_embedding_table,
    axis=1,
)

shifted_words = pd.DataFrame(
    {
        "word": training_vocabulary,
        "embedding_shift": embedding_shift,
    }
).sort_values(
    "embedding_shift",
    ascending=False,
)

shifted_words.head(12)

Large shifts may indicate useful task adaptation or overfitting. Validation and
semantic inspection are required.

# 20. Error Analysis

In [ ]:
tuned_labels = label_encoder.inverse_transform(
    tuned_ids
)

error_frame = pd.DataFrame(
    {
        "text": X_test.reset_index(drop=True),
        "actual": y_test.reset_index(drop=True),
        "predicted": tuned_labels,
        "confidence": tuned_probabilities.max(axis=1),
    }
)

error_frame["correct"] = (
    error_frame["actual"]
    == error_frame["predicted"]
)

error_frame.sort_values(
    ["correct", "confidence"],
    ascending=[True, True],
)

In [ ]:
confusion = confusion_matrix(
    y_test,
    tuned_labels,
    labels=label_encoder.classes_,
)

pd.DataFrame(
    confusion,
    index=[
        f"actual_{label}"
        for label in label_encoder.classes_
    ],
    columns=[
        f"predicted_{label}"
        for label in label_encoder.classes_
    ],
)

Error categories may include OOV-heavy examples, lexical ambiguity, weak
pooling, domain mismatch, and insufficient labeled data.

# 21. Domain Mismatch

A vector trained on general news may represent technical, medical, legal, or
dialectal words poorly.

In [ ]:
domain_mismatch = pd.DataFrame(
    [
        ("General news", "cell", "politics, prison, biology"),
        ("Technology", "cell", "mobile network, battery"),
        ("Biomedicine", "cell", "membrane, tissue, nucleus"),
        ("Finance", "charge", "payment, fee"),
        ("Technology", "charge", "battery, electricity"),
    ],
    columns=["Domain", "Word", "Likely associations"],
)

domain_mismatch

Domain adaptation options include continued pretraining, in-domain embeddings,
fine-tuning, vocabulary expansion, and subword modeling.

# 22. Embedding Dimension and Memory

Approximate memory for a float32 embedding table is:

\[
vocabulary\ size 	imes dimension 	imes 4\ bytes
\]

In [ ]:
memory_examples = []

for vocabulary_size in [
    50_000,
    200_000,
    1_000_000,
]:
    for dimension in [
        100,
        300,
    ]:
        megabytes = (
            vocabulary_size
            * dimension
            * 4
            / (1024 ** 2)
        )

        memory_examples.append(
            {
                "vocabulary_size": vocabulary_size,
                "dimension": dimension,
                "float32_memory_MB": megabytes,
            }
        )

pd.DataFrame(memory_examples).round(1)

Large embedding tables can dominate model memory, especially for multilingual
vocabularies.

# 23. Bias and Responsible Use

Pretrained embeddings inherit associations from their training corpora.

Audits should examine:

- demographic names;
- occupations;
- geographic terms;
- dialects and language varieties;
- OOV rates;
- downstream error disparities.

In [ ]:
bias_audit = pd.DataFrame(
    [
        ("Coverage", "Which groups have lower vocabulary coverage?"),
        ("Neighbors", "Do names receive stereotyped associations?"),
        ("Classification", "Do class errors differ across groups?"),
        ("Domain", "Does one domain dominate pretraining?"),
        ("Documentation", "Are corpus and limitations reported?"),
    ],
    columns=["Audit area", "Question"],
)

bias_audit

Fine-tuning may preserve, amplify, or redirect inherited bias.

# 24. Arabic and Multilingual Considerations

Arabic pretrained embeddings must address:

- clitics;
- rich morphology;
- optional diacritics;
- Alef and Ya variants;
- MSA and dialects;
- Arabizi;
- code-switching;
- corpus imbalance.

In [ ]:
arabic_forms = pd.DataFrame(
    [
        ("كتاب", "book"),
        ("الكتاب", "the book"),
        ("والكتاب", "and the book"),
        ("بالكتاب", "with/by the book"),
        ("كتابه", "his book"),
    ],
    columns=["Surface form", "Illustrative meaning"],
)

arabic_forms

Word-level coverage may be low because each surface form receives a separate
vocabulary entry. Subword models can improve coverage but do not replace
explicit linguistic analysis.

In [ ]:
arabic_toy_vectors = {
    "طبيب": np.array([1.0, 0.9, 0.1, 0.0]),
    "ممرض": np.array([0.9, 1.0, 0.1, 0.0]),
    "مستشفى": np.array([0.8, 0.8, 0.2, 0.0]),
    "بنك": np.array([0.0, 0.1, 1.0, 0.9]),
    "دفع": np.array([0.0, 0.0, 0.9, 1.0]),
}

arabic_text = "الطبيب يعمل في المستشفى"
arabic_tokens = arabic_text.split()

covered = [
    token
    for token in arabic_tokens
    if token in arabic_toy_vectors
]

print("Tokens:", arabic_tokens)
print("Directly covered:", covered)

This example illustrates how the definite article can create an OOV form even
when an unprefixed lemma exists.

# 25. Reproducibility and Reporting

Report:

- embedding source;
- corpus and license;
- language and domain;
- vocabulary size;
- vector dimension;
- normalization;
- coverage;
- OOV strategy;
- pooling strategy;
- frozen or trainable status;
- learning rate;
- random seed;
- evaluation split and metrics.

In [ ]:
import platform
import sklearn

metadata = pd.Series(
    {
        "embedding_source": "toy offline pretrained file",
        "embedding_dimension": loaded_dimension,
        "embedding_vocabulary": len(pretrained_vectors),
        "downstream_examples": len(dataset),
        "classes": dataset["label"].nunique(),
        "oov_strategy": "deterministic subword fallback",
        "random_seed": 42,
        "python_version": platform.python_version(),
        "numpy_version": np.__version__,
        "scikit_learn_version": sklearn.__version__,
    },
    name="Experiment metadata",
)

metadata

# 26. Knowledge Check

1. What is a pretrained embedding?
2. How do static and contextual embeddings differ?
3. What information appears in a text embedding file?
4. Why validate vector dimension and finite values?
5. How do token and type coverage differ?
6. What is an OOV word?
7. What are common OOV strategies?
8. What information does mean pooling discard?
9. Why use TF-IDF-weighted pooling?
10. Why compare dense embeddings with sparse TF-IDF?
11. What does freezing embeddings mean?
12. What changes during embedding fine-tuning?
13. Why can fine-tuning overfit?
14. What is domain mismatch?
15. Which Arabic properties affect embedding coverage?

# 27. Exercises

## Exercise 1 — Embedding Loader

Extend the loader to support files without a header.

## Exercise 2 — Coverage Analysis

Report coverage by class and by document.

## Exercise 3 — OOV Strategies

Compare zero, random, unknown-token, and subword OOV vectors.

## Exercise 4 — Pooling

Compare mean, maximum, and TF-IDF-weighted pooling.

## Exercise 5 — Baselines

Compare dense embeddings with unigram and word-bigram TF-IDF.

## Exercise 6 — Fine-Tuning

Compare frozen and trainable embeddings across several learning rates.

## Exercise 7 — Semantic Drift

Measure nearest neighbors before and after fine-tuning.

## Exercise 8 — Arabic Classification

Build an Arabic dataset and compare word-level and subword-aware coverage.

## Challenge Exercises

1. Add a learned unknown vector.
2. Implement attention-weighted document pooling.
3. Add probability calibration.
4. Load a real GloVe or FastText file locally.
5. Build a coverage and bias audit report.

In [ ]:
if embedding_file.exists():
    embedding_file.unlink()

print("Temporary toy embedding file removed.")

# 28. Summary and Next Lesson

In this lesson:

- pretrained embeddings were introduced as transferable vector representations;
- static and contextual embeddings were distinguished;
- a text embedding file was created, loaded, and validated;
- similarity and nearest-neighbor inspection were performed;
- token and type coverage were measured;
- OOV strategies included zero and subword-derived vectors;
- mean and TF-IDF-weighted document pooling were implemented;
- dense classifiers were compared with sparse TF-IDF;
- frozen and trainable embedding strategies were contrasted;
- a NumPy embedding classifier was initialized from pretrained vectors;
- fine-tuning updated the embedding table with task gradients;
- domain mismatch, memory, bias, and Arabic coverage were analyzed.

## Next Lesson

**Lesson 25: Sentence and Document Embeddings** introduces compositional
sentence vectors, pooling strategies, weighted averaging, supervised sentence
representations, and semantic similarity evaluation.

# References

- Mikolov, T. et al. Word2Vec literature.
- Pennington, J., Socher, R., & Manning, C. D. GloVe literature.
- Bojanowski, P. et al. FastText literature.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.
- pretrained embedding transfer, evaluation, and bias literature.